# 日次株価の前処理
READMEの手順で、このNotebookのカーネル環境にプロジェクトをインストールしてください。
少量の実データを共通APIへ渡して監査します。原本の変更・価格補完・特徴量やラベルの作成は行いません。
現在は取引カレンダーがないため、明示的な未検証モードを使います。出力は学習用データではありません。

In [ ]:
from pathlib import Path
import pandas as pd
from stock_prediction.preprocessing import (
    RAW_COLUMNS, PreprocessingConfig, preprocess_prices, run_preprocessing,
)

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'README.md').is_file() and (path / 'src' / 'stock_prediction').is_dir()
)
config = PreprocessingConfig(
    start_date='2016-04-08', end_date='2016-04-15',
    allow_unverified_calendar=True,
)


In [ ]:
raw_prices = pd.read_parquet(
    root / 'data' / 'raw' / '2016.parquet',
    columns=list(RAW_COLUMNS),
    filters=[
        ('Date', '>=', pd.Timestamp(config.start_date)),
        ('Date', '<=', pd.Timestamp(config.end_date)),
    ],
)
result = preprocess_prices(raw_prices, config=config)
result.report


In [ ]:
result.prices.head()


In [ ]:
result.quarantine.head()


In [ ]:
result.missing_by_date.head()


## 年別データの保存
CLIと同じ保存処理をNotebookから呼ぶ場合の例です。既存出力は上書きせず、必ず新しいディレクトリを指定します。
```python
from datetime import datetime, timezone
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
manifest = run_preprocessing(
    raw_dir=root / 'data' / 'raw',
    output_dir=root / 'data' / 'interim' / f'notebook_{run_id}',
    report_dir=root / 'reports' / f'notebook_{run_id}',
    config=config,
)
```
取引カレンダーを取得した後は、`preprocess_prices(..., trading_calendar=calendar)` または `run_preprocessing(..., calendar_path=path)` で指定できます。
調整済み価格は提供スナップショットであり、後日調整の時点整合性は後段で検証が必要です。

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from datetime import datetime, timezone

from src.stock_prediction.preprocessing import PreprocessingConfig, run_preprocessing


snapshot_dir = "../data/raw"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
preprocess_config = PreprocessingConfig()
preprocess_manifest = run_preprocessing(
    raw_dir=snapshot_dir + "/prices",
    calendar_path=snapshot_dir + "/trading_calendar.parquet",
    output_dir=root + "/data/interim" + f"/prices_{run_id}",
    report_dir=root + "/reports" + f"/preprocessing_{run_id}",
    config=preprocess_config,
)

preprocess_manifest["status"], preprocess_manifest["output_rows"]

TypeError: unsupported operand type(s) for /: 'str' and 'str'